# FastAPI ML-сервис: быстрый прогон
Пошаговая проверка `/forward`, `/history`, `/stats`. Требуется запущенный Postgres/Chroma и зависимости.

## 0. Предусловия
1. Активировать venv.
2. Применить миграции: `alembic upgrade head`.
3. Запустить сервис: `uvicorn ml_service.main:app --host 0.0.0.0 --port 8000`.
4. Выполнить ячейки ниже.

In [0]:
import os, time, base64, json
import requests

BASE_URL = os.getenv('BASE_URL', 'http://localhost:8000')
JWT_SECRET = os.getenv('JWT_SECRET', 'devsecret')  # тот же, что в env сервиса

# Сгенерировать admin JWT (роль admin). Можно подставить свой токен.
try:
    import jwt
    ADMIN_TOKEN = jwt.encode({'role': 'admin', 'iat': int(time.time())}, JWT_SECRET, algorithm='HS256')
except Exception as e:
    ADMIN_TOKEN = None
    print('Не удалось сгенерировать JWT автоматически:', e)

print('BASE_URL =', BASE_URL)
print('ADMIN_TOKEN есть' if ADMIN_TOKEN else 'ADMIN_TOKEN не сгенерирован')


## 1. Текстовый запрос к /forward
Используем ключ `text` (опционально `model`).

In [0]:
payload = {'text': 'Что такое фонд?', 'model': 'deepseek'}
resp = requests.post(f"{BASE_URL}/forward", json=payload, timeout=60)
print('status', resp.status_code)
print(resp.text[:500])


In [0]:
# Дополнительный текстовый вопрос
payload = {'text': 'Расскажи кратко, что такое облигация', 'model': 'deepseek'}
resp = requests.post(f"{BASE_URL}/forward", json=payload, timeout=60)
print('status', resp.status_code)
print(resp.text[:500])


## 2. multipart (демо с картинкой)


In [0]:
png_bytes = base64.b64decode('iVBORw0KGgoAAAANSUhEUgAAAA4AAAAOCAYAAAAfSC3RAAAAHUlEQVR42mNgGAWjYBSMglEwCkb9D4YGhgYGBgAAAwCpkQo4h4Gv1gAAAABJRU5ErkJggg==')
files = {'image': ('sample.png', png_bytes, 'image/png')}
headers = {'X-Model': 'demo'}
resp = requests.post(f"{BASE_URL}/forward", files=files, headers=headers, timeout=60)
print('status', resp.status_code)
print(resp.text[:200])


## 3. История


In [0]:
resp = requests.get(f"{BASE_URL}/history", timeout=30)
print('status', resp.status_code)
print(resp.json() if resp.ok else resp.text)


## 4. Статистика (admin JWT)


In [0]:
headers = {}
if ADMIN_TOKEN:
    headers['Authorization'] = f'Bearer {ADMIN_TOKEN}'
resp = requests.get(f"{BASE_URL}/stats", headers=headers, timeout=30)
print('status', resp.status_code)
print(resp.json() if resp.ok else resp.text)


## 5. Очистка истории (admin JWT)


In [0]:
headers = {}
if ADMIN_TOKEN:
    headers['Authorization'] = f'Bearer {ADMIN_TOKEN}'
resp = requests.delete(f"{BASE_URL}/history", headers=headers, timeout=30)
print('status', resp.status_code)
print(resp.text)
